# 1.0 - Data Retrivial & Data Preparation

In [22]:
# import
import os
import json
import pandas as pd

In [23]:
# setup
input_dir = '../data/raw'
df = pd.DataFrame()

In [24]:
# read ndjson data
for _file in os.listdir(input_dir):
    temp_df = pd.read_json(os.path.join(input_dir, _file), lines = True)
    df = pd.concat([df, temp_df]).reset_index()

In [25]:
# remove useless columns
df = df.drop(columns = ['level_0', 'index'], axis = 1)

In [26]:
# define row id
df['index'] = df.index

In [28]:
df = df.sample(5)
df

,domain,object_id,object_type,object_title,object_text,object_post_type,object_url,object_abstract,object_featured_image,object_pub_date,object_post_image,object_post_video,object_post_link,object_post_category,object_post_tag,domain_category,links_words_rate,index
5273,www.pianetamamma.it,pm_25370,post,"Gli uomini non dicono, ma non sempre",\r\n\tPausa pranzo. Sto al bar con due collegh...,article,https://www.pianetamamma.it/la-famiglia/divent...,"Quello che gli uomini non dicono, ma non sempr...","{""image_id"": ""15647465"", ""image_alt"": """", ""ima...",2018-11-26 15:59:11,"[{""image_id"": ""15647465"", ""image_alt"": """", ""im...",[],"[{""link_url"": ""https://www.pianetamamma.it/il-...","[{""category_id"": ""2"", ""category_url"": ""la-fami...",null,parenting,1.353383,5273
19954,www.nostrofiglio.it,Nf_24985,post,Lavoretti di carta da fare con i bambini,Volete trascorre un po' di tempo piacevole con...,article,https://www.nostrofiglio.it/famiglia/lavoretti...,Se desiderate trascorrere un po' di tempo piac...,"{""image_id"": ""4755157"", ""image_alt"": """", ""imag...",2019-05-14 16:42:00,"[{""image_id"": ""4755157"", ""image_alt"": """", ""ima...",[],"[{""link_url"": ""https://www.nostrofiglio.it/gio...","[{""category_id"": ""10"", ""category_url"": ""www.no...",null,parenting,0.615385,19954
12041,www.nostrofiglio.it,Nf_13835,post,Frasi e citazioni SPECIALI per la festa del papà,\n\n\n\n\n\n\n\n\n\n\n\n\n\n,gallery,https://www.nostrofiglio.it/feste/festa-del-pa...,Tra pochissimi giorni sarà la festa di tutti i...,"{""image_id"": ""5218800"", ""image_alt"": """", ""imag...",2023-03-02 10:23:00,"[{""image_id"": ""5218800"", ""image_alt"": """", ""ima...",[],[],"[{""category_id"": ""343235"", ""category_url"": ""ww...",null,parenting,0.000000,12041
15488,www.nostrofiglio.it,nf_18344,post,10 musei in Italia e nel mondo in cui è possib...,Esistono molti musei in diversi luoghi del mon...,article,https://www.nostrofiglio.it/famiglia/musei-in-...,"Chi non ricorda la famosa commedia ""Una notte ...","{""image_id"": ""4743653"", ""image_alt"": """", ""imag...",2018-04-11 12:54:00,"[{""image_id"": ""0"", ""image_alt"": ""False"", ""imag...",[],"[{""link_url"": ""https://www.nostrofiglio.it/tag...","[{""category_id"": ""10"", ""category_url"": ""famigl...",null,parenting,1.484480,15488
22327,www.nostrofiglio.it,Nf_259102,post,Gli asteroidi spiegati ai bambini: una piccola...,"Il cielo, la sua innegabile infinità e i suoi ...",article,https://www.nostrofiglio.it/bambino/istruzione...,Il 30 giugno di ogni anno ricorre la Giornata ...,"{""image_id"": ""0"", ""image_alt"": null, ""image_ur...",2023-06-29 16:05:00,[],"[{""video_id"": ""5339347"", ""video_url"": ""https:/...","[{""link_url"": ""https://www.nostrofiglio.it/bam...","[{""category_id"": ""343216"", ""category_url"": ""ww...",null,parenting,0.661001,22327


In [29]:
# define json structure for requests towards OpenAI API
json_list = list()
json_requests = {
    'method' : 'POST',
    'url' : '/v1/chat/completions'
}

MODEL = os.getenv('MODEL')

for index, row in df.iterrows():
    json_requests['custom_id'] = f'doc_{index}'
    json_requests['body'] = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": f"""Modifica il seguente testo in lingua italiana, in modo tale da permettere una corretta divisione del testo in frasi. Sostituisci i punti che dividono le frasi con il simbolo |.
Puoi eliminare tutte le frasi che iniziano con 'LEGGI ANCHE:' oppure 'Fonte:'
Testo originale:
{row['object_text']}

Testo corretto:"""
            }
        ]
    }
    json_list.append(json_requests.copy())

In [31]:
# save jsonl file (one json per line)
with open('../data/processed/requests.jsonl', 'w') as f:
    for item in json_list:
        json.dump(item, f)
        f.write('\n')

In [32]:
# save data to parquet
df.to_parquet('../data/processed/full_articles_data.parquet', engine = 'pyarrow', index = False)

---
# Cost Estimation (ChatGPT 3.5-Turbo)

In [33]:
df['len'] = df['object_text'].apply(lambda x: len(x))

In [34]:
# define prompt
prompt = """Modifica il seguente testo in lingua italiana, in modo tale da permettere una corretta divisione del testo in frasi. Sostituisci i punti che dividono le frasi con il simbolo |.
Puoi eliminare tutte le frasi che iniziano con 'LEGGI ANCHE:' oppure 'Fonte:'
Testo originale:

Testo corretto:"""

In [ ]:
input_price = (df['len'].sum() + len(df)*len(prompt)) / 4 * .5 / 1_000_000
output_price = df['len'].sum() / 4 * 1.5 / 1_000_000

print(f"Average charcaters per article: {int(df['len'].sum()/len(df))}")
print(f"Estimated tokens: {int(df['len'].sum()/4)}")
print(f"Estimated pricing (input): {round(input_price, 2)}€")
print(f"Estimated pricing (output): {round(output_price, 2)}€")